In [10]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import torchvision.transforms as transforms
import torch.utils.data as data

## Upload dataset, Remove nulls

In [ ]:
df = pd.read_pickle('dataset5000.pkl')
df = df.dropna().drop(['NAME','Scale','Activity','Category','ID'], axis=1)
#convert the columns upper neck_X upper neck_Y  head top_X  head top_Y to int64
df = df.astype({'upper neck_X': 'int64', 'upper neck_Y': 'int64', 'head top_X': 'int64', 'head top_Y': 'int64'})

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4982 entries, 0 to 7853
Data columns (total 33 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   r ankle_X     4982 non-null   int64 
 1   r ankle_Y     4982 non-null   int64 
 2   r knee_X      4982 non-null   int64 
 3   r knee_Y      4982 non-null   int64 
 4   r hip_X       4982 non-null   int64 
 5   r hip_Y       4982 non-null   int64 
 6   l hip_X       4982 non-null   int64 
 7   l hip_Y       4982 non-null   int64 
 8   l knee_X      4982 non-null   int64 
 9   l knee_Y      4982 non-null   int64 
 10  l ankle_X     4982 non-null   int64 
 11  l ankle_Y     4982 non-null   int64 
 12  pelvis_X      4982 non-null   int64 
 13  pelvis_Y      4982 non-null   int64 
 14  thorax_X      4982 non-null   int64 
 15  thorax_Y      4982 non-null   int64 
 16  upper neck_X  4982 non-null   int64 
 17  upper neck_Y  4982 non-null   int64 
 18  head top_X    4982 non-null   int64 
 19  head top_Y 

## Contruct the Dataset Object

In [11]:
keypoint_columns = [col for col in df.columns if col != 'image'] #todas as coluunas 
keypoints = df[keypoint_columns].values.astype(np.float32)
 
keypoints = keypoints / 224.0 #normaliza os pontos para 0-1

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

class PoseDataset(data.Dataset):
    def __init__(self, df):
        self.df = df
        self.images = df['image'].tolist()
        self.keypoints = keypoints
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_array = self.images[idx]
        img_tensor = transform(img_array) #converter para pythorch
        kp_tensor = torch.tensor(self.keypoints[idx])
        return img_tensor, kp_tensor

## Spliting for test and train 

In [12]:
dataset = PoseDataset(df)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) -train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = data.DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = data.DataLoader(val_dataset, batch_size=32, shuffle=False)

## Defining the model

In [13]:


class PoseNet(nn.Module):
    def __init__(self):
        super(PoseNet, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 112x112
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 56x56
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 28x28
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 32) 
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

## Training the model

In [27]:
model = PoseNet()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001) 

In [30]:
num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    
    for images, keypoints in train_loader:
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, keypoints)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {train_loss/len(train_loader):.4f}")
    
    if(epoch+1) % 1 == 0:
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for images, keypoints in val_loader:
                outputs = model(images)
                loss = criterion(outputs, keypoints)
                val_loss += loss.item()
        print(f"Val Loss: {val_loss/len(val_loader):.4f}")
    


Epoch [1/20], Loss: 0.0049
Val Loss: 0.0400
Epoch [2/20], Loss: 0.0041
Val Loss: 0.0406
Epoch [3/20], Loss: 0.0034
Val Loss: 0.0402
Epoch [4/20], Loss: 0.0028
Val Loss: 0.0407


KeyboardInterrupt: 

## Training logs
lr=0.01; epoch: 10; Loss: 0.0014; Val Loss: 0.0441

In [29]:
torch.save(model.state_dict(), "posev0402.pth")


In [ ]:
model.load_state_dict(torch.load("pose_net.pth"))
model.eval()